In [ ]:
import tensorflow as tf 
import numpy as np 
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.activations import relu, softmax, linear, sigmoid 
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt

In [ ]:
# def valid_isbn():
#     x = np.random.randint(0, 10, 9)
#     weights = np.arange(10, 1, -1)
#     # print(weights)
#     # print(x)
#     summation = np.dot(weights, x)
#     # print(summation)
#     last_digit = (11 - (summation % 11))%11
#     return np.append(x, last_digit)
def valid_isbn():
    while True:
        digits = np.random.randint(0, 10, 9)
        weights = np.arange(10, 1, -1)
        s = np.dot(weights, digits)
        check = (11 - (s % 11)) % 11
        if check < 10:
            return np.append(digits, check)

In [ ]:
def is_valid(isbn):
    weights = np.arange(10, 0, -1)
    return np.dot(weights, isbn) % 11 == 0


In [ ]:
# def invalid_isbn(isbn):
#     while True:
#         invalid = isbn.copy()

#         if np.random.rand() < 0.5:
#             pos = np.random.randint(0, 10)
#             new = np.random.randint(0, 10)
#             invalid[pos] = new
#         else:
#             pos = np.random.randint(0, 9)
#             invalid[pos], invalid[pos+1] = invalid[pos+1], invalid[pos]

#         if not is_valid(invalid):
#             return invalid
def invalid_isbn():
    while True:
        digits = np.random.randint(0, 10, 10)
        if not is_valid(digits):
            return digits

In [ ]:
isbn = valid_isbn()
print(isbn)
# print(invalid_isbn(isbn))

In [ ]:
def create_dataset(n):
    X = []
    y = []
    weights = np.arange(10, 0, -1)

    for i in range(n):
        v = valid_isbn()
        checksum = np.dot(weights, v)
        r_v = checksum % 11
        X.append(np.concatenate([v, [checksum, r_v]]))
        y.append(1)


        iv = invalid_isbn()
        checksumi = np.dot(weights, iv)
        r_iv = checksumi % 11
        X.append(np.concatenate([iv, [checksumi, r_iv]]))
        y.append(0)

    return np.array(X), np.array(y)


In [ ]:
# x = data[:, :10]
# y = data[:, 10]

In [ ]:

X, y = create_dataset(10000)
X.shape

In [ ]:
x_train, x_, y_train, y_ = train_test_split(X, y,test_size = 0.4, shuffle =True, random_state = 42)
x_dev, x_test, y_dev, y_test = train_test_split(x_, y_, test_size= 0.5, shuffle=True, random_state=42)

In [ ]:
x_test.shape

In [ ]:
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train)
x_dev = scaler.transform(x_dev)

x_test = scaler.transform(x_test)

In [ ]:
tf.random.set_seed(1234)
model = Sequential([
    Dense(128, activation='relu', input_shape=(12,)),
    Dropout(0.2), 
    Dense(128, activation='relu'),
    Dense(64, activation='relu'),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

In [ ]:
# print(x_train.shape)
# print(y_train.shape)
# print(y_train[:10])

In [ ]:
print(np.mean(y_train))


In [ ]:
model.compile(
    loss="binary_crossentropy", 
    optimizer=tf.keras.optimizers.Adam(0.0005),
    metrics=["accuracy"]
)
history = model.fit(
    x_train, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(x_dev, y_dev) 
)


In [ ]:
y_pred_prob = model.predict(x_test)
y_pred = (y_pred_prob >= 0.5).astype(int)

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test)

In [ ]:
print(classification_report(y_test, y_pred))

In [ ]:
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.legend()
plt.show()

In [ ]:
# تمرین 3.5 کتاب 
first = "0131653326"
second = "0139241014"
def male_array(s):
    return np.array([int(d) for d in s])
    
def prepare_input(isbn):
    weights = np.arange(10, 0, -1)
    c = np.dot(weights, isbn)
    r = c % 11
    data = np.concatenate([isbn, [c, r]]).reshape(1, -1)
    return scaler.transform(data)

test_samples = [
    male_array(first),
    male_array(second)
]
labels = ["first", "second"]
    

for sample, label in zip(test_samples, labels):
    processed = prepare_input(sample)
    pred = model.predict(processed)
    print(f"{label}, predicted : {pred[0][0]:.4f}")